# Qlib Research Guide (Google Colab Friendly)
This notebook is a kid-friendly ("explain like I'm 5") tour of [Qlib](https://github.com/microsoft/qlib). It walks through installing Qlib, loading data, building models, forecasting, backtesting, and analyzing performance. Every code cell is meant to run top-to-bottom in Google Colab.

**Theme:** We use the US S&P 500 daily data so you can try things quickly in Colab.

## Part 1 — Overview of Qlib
* **What is Qlib?** Qlib is like a big toy box full of tools for people who study stocks. It stores market data, builds signals (called *factors*), trains models that guess future prices, and checks how good those guesses are.
* **Why do quant researchers use it?** Because Qlib saves time. It already knows how to fetch data, clean it, train many models, and test trading ideas.
* **Key words (kid style):**
  * **Market data:** the numbers for prices/volumes of many companies (like scores on many toys).
  * **Factors/features:** smart hints made from the data (like “this toy is shiny,” “this toy is big”).
  * **Model:** a robot brain that looks at hints and makes a guess.
  * **Alpha/signal/score:** the robot’s guess about which stock might do well.
  * **Backtesting:** pretending to trade in the past to see if the idea works.

## Part 2 — Installation & Setup
Run the next cells in Google Colab. They install Qlib from GitHub (latest), download US data, and initialize the engine.

In [ ]:
# Install from GitHub to get the newest Qlib (works in Colab)!
pip install --quiet git+https://github.com/microsoft/qlib.git


In [ ]:
# Imports used throughout the notebook
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import qlib
from qlib.constant import REG_US
from qlib.utils import init_instance_by_config
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import DataHandlerLP
from qlib.contrib.model.gbdt import LGBModel
from qlib.contrib.model.linear import LinearModel
from qlib.contrib.model.pytorch_gru import GRUModel
from qlib.contrib.model.pytorch_gats import GATSModel
from qlib.contrib.strategy import TopkDropoutStrategy
from qlib.contrib.executor import SimulatorExecutor
from qlib.contrib.evaluate import backtest, risk_analysis
from qlib.contrib.report import analysis_position

plt.style.use('seaborn-v0_8')


In [ ]:
# Download US data set to the user directory (~/.qlib/qlib_data)
!python -m qlib.run.get_data --target_dir ~/.qlib/qlib_data/us_data --region us --download_mode auto


In [ ]:
# Choose which market to work with (US S&P 500 by default) and initialize Qlib
MARKET = "sp500"          # U.S. S&P 500 universe
BENCHMARK = "SP500"       # Benchmark for backtests
DATA_PATH = os.path.expanduser("~/.qlib/qlib_data/us_data")

qlib.init(provider_uri=DATA_PATH, region=REG_US, expression_cache=None, dataset_cache=None)
print("Qlib ready! Version:", qlib.__version__)
print("Data path exists:", os.path.exists(DATA_PATH))


## Part 3 — Data Processing
We now set up **DataHandler**, **Dataset**, features, and labels.

* **DataHandler (kid story):** Imagine a friendly robot that cleans your toy room. It picks which toys (stocks) to keep, washes them (fill missing values), and labels them so you know where they belong. Qlib's `DataHandlerLP` does this for market data.
* **Processor pipeline:** Tiny cleaners that run in order (drop bad labels, normalize, fill blanks).
* **Dataset:** Connects the handler recipe with calendar splits (train/valid/test).

In [ ]:
# Configuration for the data handler (feature engineering + labeling)
handler_config = {
    "class": "DataHandlerLP",
    "module_path": "qlib.contrib.data.handler",
    "kwargs": {
        "start_time": "2017-01-01",
        "end_time": "2022-12-31",
        "fit_start_time": "2017-01-01",
        "fit_end_time": "2020-12-31",
        "instruments": MARKET,
        # Features: simple price/volume + rolling stats
        "feature": [
            "$close",
            "$open",
            "$high",
            "$low",
            "$volume",
            "Ref($close, 1)",
            "Ref($close, 5)",
            "Mean($close, 5)",
            "Mean($close, 20)",
            "Std($close, 5)",
            "Std($close, 20)",
            "Mean($volume, 20)",
            "$close / Ref($close, 5) - 1",
        ],
        # Label: 2-day future return (tomorrow vs. the next day)
        "label": ["Ref($close, -2) / Ref($close, -1) - 1"],
        # Learn-time processors: drop bad labels, normalize features & labels per day
        "learn_processors": [
            {"class": "DropnaLabel"},
            {"class": "CSRankNorm", "kwargs": {"fields_group": "label"}},
            {"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}},
            {"class": "Fillna", "kwargs": {"fields_group": "feature"}},
        ],
        # Infer-time processors: same normalization for inference
        "infer_processors": [
            {"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}},
            {"class": "Fillna", "kwargs": {"fields_group": "feature"}},
        ],
    },
}

# Segment the calendar into train/valid/test
segments = {
    "train": ("2017-01-01", "2019-12-31"),
    "valid": ("2020-01-01", "2020-06-30"),
    "test": ("2020-07-01", "2022-12-31"),
}

# Build the dataset: DatasetH glues handler + segments together
dataset = DatasetH(handler=handler_config, segments=segments)
print(dataset)


In [ ]:
# Load a tidy DataFrame for each split
train_df = dataset.prepare("train")
valid_df = dataset.prepare("valid")
test_df = dataset.prepare("test")
print("Train rows:", len(train_df))
train_df.head()


In [ ]:
# Check the schema: features vs labels
feature_cols = [c for c in train_df.columns if c.startswith("feature::")]
label_cols = [c for c in train_df.columns if c.startswith("label::")]
print("Number of features:", len(feature_cols))
print("Number of labels:", len(label_cols))


In [ ]:
# Pick one instrument and plot its close price and a label example
sample_inst = train_df.index.get_level_values("instrument")[0]
plot_df = train_df.xs(sample_inst, level="instrument")

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(plot_df.index, plot_df['feature::$close'], label='Close Price', color='blue')
ax1.set_ylabel('Price')
ax1.legend(loc='upper left')
ax1.set_title(f"Sample instrument: {sample_inst}")

ax2 = ax1.twinx()
ax2.plot(plot_df.index, plot_df['label::Ref($close, -2) / Ref($close, -1) - 1'], label='Future 2-day return', color='green', alpha=0.6)
ax2.set_ylabel('Future return')
ax2.legend(loc='upper right')
plt.show()


## Part 4 — Modeling
Qlib has a **model zoo**. We try four: Linear, LightGBM, GRU (RNN), and GAT (graph attention).

* **Hyperparameters (kid words):** knobs and sliders that change how the model learns (like how many LEGO blocks high your tower is, or how fast you build it).
* **Train/valid/test split:** we teach on *train*, check ourselves on *valid*, and finally score on *test*.


In [ ]:
# Helper: train a model and get predictions on all splits
def train_and_predict(model, dataset, name):
    print(f"Training {name}...")
    model.fit(dataset)
    pred = model.predict(dataset)
    print(f"{name} done. Prediction sample:
", pred.head())
    return pred

# 1) Linear baseline
linear_model = LinearModel()
linear_pred = train_and_predict(linear_model, dataset, "LinearModel")

# 2) Gradient-boosted trees (LightGBM)
lgb_model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=64,
    n_estimators=200,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
)
lgb_pred = train_and_predict(lgb_model, dataset, "LGBModel")

# 3) GRU deep model
gru_model = GRUModel(
    d_feat=dataset.handler.get_feature_dim(),
    hidden_size=64,
    num_layers=2,
    drop_prob=0.1,
    n_epochs=5,
    lr=1e-3,
    batch_size=512,
    early_stop=2,
    metric="loss",
    loss="mse",
)
gru_pred = train_and_predict(gru_model, dataset, "GRUModel")

# 4) Graph Attention Network (GAT)
gat_model = GATSModel(
    d_feat=dataset.handler.get_feature_dim(),
    hidden_size=64,
    num_layers=2,
    n_epochs=5,
    lr=1e-3,
    batch_size=512,
    drop_prob=0.1,
    metric="loss",
    loss="mse",
)
gat_pred = train_and_predict(gat_model, dataset, "GATSModel")


In [ ]:
# Pick one model for the rest of the demo
best_pred = lgb_pred
best_pred.name = "score"

# Extract the aligned label for evaluation
label = dataset.prepare("test", col_set="label")
print(best_pred.head())
print(label.head())


## Part 5 — Forecasting / Inference
Now we use the trained model to make signals (scores), save them, and visualize them.

* **Alpha / signal / score (kid words):** The model's treasure map telling us which stocks might shine. Bigger score = stronger guess.


In [ ]:
# Save predictions to disk (optional)
pred_path = "./predictions.csv"
best_pred.to_csv(pred_path)
print("Saved to", pred_path)

# Plot prediction distribution
plt.figure(figsize=(6,4))
sns.histplot(best_pred.values, bins=40, kde=True)
plt.title("Prediction score distribution")
plt.show()


## Part 6 — Portfolio & Backtest
We turn scores into a pretend portfolio, run a backtest, and draw the pretend money curve.

* **Execution model:** tells how we trade each day (here: a simple simulator that acts instantly).
* **Position management:** how many stocks we hold and when we drop some.
* **Sharpe ratio (kid words):** how smooth the ride is (reward per bump). Bigger is smoother reward.
* **Drawdown:** biggest slide downward from a peak (how scary the ride gets).
* **Turnover:** how many toys (stocks) we swap out each day.


In [ ]:
# Build strategy from the prediction scores
strategy_config = {
    "class": "TopkDropoutStrategy",
    "module_path": "qlib.contrib.strategy",
    "kwargs": {
        "signal": best_pred,
        "topk": 10,
        "n_drop": 2,
    },
}
strategy = init_instance_by_config(strategy_config)

# Simple simulator executor
executor_config = {
    "class": "SimulatorExecutor",
    "module_path": "qlib.contrib.executor",
    "kwargs": {
        "time_per_step": "day",
        "generate_report": True,
    },
}
executor = init_instance_by_config(executor_config)

portfolio_metrics, indicator = backtest(
    strategy=strategy,
    executor=executor,
    benchmark=BENCHMARK,
    return_type="report",
)
print("Backtest metrics keys:", portfolio_metrics.keys())


In [ ]:
# Plot equity curve from backtest
report_df = portfolio_metrics['report']
plt.figure(figsize=(10,4))
plt.plot(report_df['datetime'], report_df['return'].cumsum(), label='Strategy cumulative return')
plt.title('Backtest Equity Curve (Pretend Money Growth)')
plt.xlabel('Date')
plt.ylabel('Cumulative return')
plt.legend()
plt.show()


In [ ]:
# Compute risk analysis (includes IC & ICIR when signals and labels align)
risk_report = risk_analysis(report_df, benchmark=BENCHMARK, freq='day')
print(risk_report)

position_report = analysis_position.report_graph(indicator)
position_report.head()


## Part 7 — Performance Analysis
We inspect metrics:
* **IC (Information Coefficient):** how well scores line up with future returns (like guessing which friend wins a race).
* **ICIR:** stability of IC over time (steady guessing skill).
* **Cumulative/annualized return:** how much pretend money grows.
* **Risk metrics (volatility, max drawdown, Sharpe):** how bumpy the ride is.
* **Feature importance:** which hints mattered most (for tree models).


In [ ]:
# Feature importance for the LightGBM model
fi = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi.head(20).plot(kind='barh', figsize=(6,6))
plt.title('Top feature importance (LightGBM)')
plt.xlabel('Importance')
plt.show()


In [ ]:
# Confusion-matrix-like check: bucket predictions and compare returns
pred_label = pd.concat([best_pred.rename('score'), label.iloc[:,0].rename('label')], axis=1).dropna()
pred_label['bucket'] = pd.qcut(pred_label['score'], 5, labels=False)
bucket_stats = pred_label.groupby('bucket')['label'].mean()
print(bucket_stats)
bucket_stats.plot(kind='bar', figsize=(6,4))
plt.title('Average future return by score bucket')
plt.xlabel('Bucket (higher = stronger score)')
plt.ylabel('Average future return')
plt.show()


## Part 8 — Advanced Qlib Features
* **Online serving:** save a trained model and load it later to serve fresh scores.
* **Optimization:** tiny hyperparameter search loop example.
* **Workflow pipelines:** combine handlers, datasets, models, and backtests as building blocks.
* **Custom DataHandlers / models:** templates to extend Qlib.


In [ ]:
# Save & reload a trained model (serving-style)
model_path = "./lgb_model.pkl"
lgb_model.save(model_path)
print("Saved model to", model_path)

reloaded = LGBModel()
reloaded.load(model_path)
print("Reloaded model ready:", reloaded)


In [ ]:
# Hyperparameter search example (very small for demo)
search_spaces = {
    "learning_rate": [0.01, 0.05],
    "num_leaves": [31, 63],
}
best_params = None
best_ic = -np.inf

for lr in search_spaces['learning_rate']:
    for nl in search_spaces['num_leaves']:
        temp_model = LGBModel(learning_rate=lr, num_leaves=nl, n_estimators=50)
        temp_model.fit(dataset)
        pred = temp_model.predict(dataset)
        joined = pd.concat([pred.rename('score'), label.iloc[:,0].rename('label')], axis=1).dropna()
        ic = joined[['score','label']].corr().iloc[0,1]
        print(f"lr={lr}, num_leaves={nl}, IC={ic:.4f}")
        if ic > best_ic:
            best_ic = ic
            best_params = {"learning_rate": lr, "num_leaves": nl}

print("Best params:", best_params, "with IC", best_ic)


In [ ]:
# Template for a custom DataHandler
from qlib.data.dataset.handler import DataHandlerLP as BaseHandler

class MyCustomHandler(BaseHandler):
    def get_feature_config(self):
        return ["$close", "Mean($close, 5)"]

    def get_label_config(self):
        return ["Ref($close, -1) / $close - 1"]

# Example creation (not used elsewhere in the notebook)
custom_dataset = DatasetH(handler={"class": "MyCustomHandler", "module_path": "__main__"}, segments=segments)
custom_dataset


In [ ]:
# Template for a custom model
from qlib.model.base import Model

class MyTinyModel(Model):
    def fit(self, dataset):
        # pretend to train; normally you would use dataset.prepare()
        self.mean_label = dataset.prepare("train", col_set="label").mean().values[0]

    def predict(self, dataset):
        # predict the same value for all rows (toy example)
        label_index = dataset.prepare("train", col_set="label").index
        return pd.Series(self.mean_label, index=label_index)

# Demonstrate creation
mtm = MyTinyModel()
mtm


## Part 9 — Summary & Next Steps
**What we did:** installed Qlib, downloaded US data, built features/labels, trained Linear/LGBM/GRU/GAT models, made forecasts, ran a backtest, plotted equity, and read metrics like IC and feature importance.

**Explore next:**
* Try different universes (e.g., `csi300` for China A-shares) by changing `MARKET` and `BENCHMARK`.
* Add more features (technical, fundamental) to test new ideas.
* Explore Qlib docs: https://qlib.readthedocs.io/
* Contribute: https://github.com/microsoft/qlib/pulls
* Try more datasets: NASDAQ or full US data by adjusting the download command.
